# 05 — Chain of Thought + Structured Output

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/05_cot_structured.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.**
1. Ver cómo CoT mejora respuestas de razonamiento.
2. Generar JSON parseable directamente desde el modelo, listo para integrar con código.


In [ ]:
%pip install --quiet groq


In [ ]:
import os
import json
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODEL = "qwen/qwen3-32b"

def chat(messages, **kwargs):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **kwargs,
        # reasoning_effort="none",  # descomentar para que Qwen3 no devuelva <think>...</think>
    )
    return resp.choices[0].message.content


## Parte 1 — Chain of Thought

Problema típico de sistemas: cuántos GB de logs genera tu API.


In [ ]:
PROBLEMA = '''
Tenemos un sistema con 1.000.000 de usuarios registrados.
El 0,3% son usuarios premium.
Cada usuario premium genera en promedio 15 eventos por día.
Cada evento se almacena como una fila de 512 bytes en una tabla de logs.

Pregunta: ¿cuántos GB de logs de eventos premium se generan por SEMANA?
'''
print(PROBLEMA)


### Sin CoT — respuesta directa


In [ ]:
print(chat(
    [{"role": "user", "content": PROBLEMA}],
    temperature=0.1,
))


### Con CoT — forzando paso a paso


In [ ]:
SYSTEM_COT = (
    "Sos un ingeniero de sistemas. Resolvé los problemas paso a paso, "
    "mostrando cada operación con su unidad antes de dar la respuesta final. "
    "Terminá con una línea 'Respuesta: <valor>'."
)

print(chat(
    [
        {"role": "system", "content": SYSTEM_COT},
        {"role": "user", "content": PROBLEMA},
    ],
    temperature=0.1,
))


### Verificación numérica

Calculamos a mano para comparar.


In [ ]:
usuarios_premium = 1_000_000 * 0.003
eventos_dia      = usuarios_premium * 15
eventos_semana   = eventos_dia * 7
bytes_semana     = eventos_semana * 512
gb_semana        = bytes_semana / (1024**3)

print(f"Premium:           {usuarios_premium:>15,.0f} usuarios")
print(f"Eventos por semana:{eventos_semana:>15,.0f}")
print(f"Bytes por semana:  {bytes_semana:>15,.0f}")
print(f"GB por semana:     {gb_semana:>15.3f}")


## Parte 2 — Structured Output (JSON)

Forzamos al modelo a devolver JSON parseable. Se usa muchísimo en producción para integrar LLMs con código tradicional.


In [ ]:
SYSTEM_JSON = '''Analizás requerimientos de software.
Respondés SOLO con un JSON válido con esta estructura exacta:
{
  "tipo": "funcional" | "no_funcional" | "restriccion",
  "prioridad": "alta" | "media" | "baja",
  "componente": string,
  "resumen": string (máx 15 palabras)
}
No incluyas nada más que el JSON. Sin markdown, sin explicaciones.'''

REQUERIMIENTOS = [
    "El sistema debe responder en menos de 200ms al 95% de las solicitudes bajo carga normal.",
    "Los usuarios pueden recuperar su contraseña por email.",
    "Toda la información sensible debe almacenarse cifrada en reposo (AES-256).",
]

for req in REQUERIMIENTOS:
    raw = chat(
        [
            {"role": "system", "content": SYSTEM_JSON},
            {"role": "user", "content": req},
        ],
        temperature=0.1,
    )

    # parse defensivo: a veces el modelo mete ```json ... ```
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        data = json.loads(clean)
        print(f"OK -> {req}")
        for k, v in data.items():
            print(f"    {k}: {v}")
    except json.JSONDecodeError as e:
        print(f"Error parseando: {e}")
        print(f"  raw: {raw}")
    print()


## Tips para production

- **Tolerá errores de parseo**: usá `json-repair` o un retry con un mensaje "el JSON anterior estaba mal".
- **Validá con un schema** (pydantic, jsonschema). El modelo se equivoca: si el schema no se cumple, descartá la respuesta.
- **`response_format={"type": "json_object"}`**: muchos modelos (incluido Qwen vía Groq) tienen un modo JSON forzado. Probalo.
- **Para tareas de razonamiento + JSON**: pedí primero el razonamiento (en un campo `reasoning`) y después la respuesta. Mejora calidad.
